In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import zipfile

# 1. הגדרת נתיבים (מותאם למחברת בתוך תיקיית notebooks)
BASE_DIR = Path(os.getcwd()).parent
ZIP_PATH = BASE_DIR / "data" / "raw" / "rais_anonymized.zip"
DATA_DIR = BASE_DIR / "data" / "processed_parquet"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. חילוץ שאלוני ה-STAI וה-PANAS ישירות מקובץ ה-ZIP
print("מחלץ שאלונים מקובץ ה-ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open('rais_anonymized/scored_surveys/stai.csv') as f:
        stai = pd.read_csv(f)
    with z.open('rais_anonymized/scored_surveys/panas.csv') as f:
        panas = pd.read_csv(f)

# שמירה לפורמט Parquet לעבודה מהירה בהמשך
stai.to_parquet(DATA_DIR / "stai.parquet")
panas.to_parquet(DATA_DIR / "panas.parquet")

# 3. נרמול תאריכים
stai['date'] = pd.to_datetime(stai['submitdate'], errors='coerce').dt.normalize()
panas['date'] = pd.to_datetime(panas['submitdate'], errors='coerce').dt.normalize()

# 4. יצירת מילון המזהים (new_id) מתוך השאלונים עצמם
# שולפים את כל המזהים הארוכים (הייחודיים) מתוך שני השאלונים
unique_long_ids = pd.concat([stai['user_id'], panas['user_id']]).dropna().unique()

id_mapping = pd.DataFrame({
    'id': unique_long_ids,
    'new_id': range(1, len(unique_long_ids) + 1)
})

# 5. טיפול בטבלאות כדי ליצור את stai_clean ו-panas_clean
stai_clean = stai.rename(columns={'user_id': 'id'})
stai_clean = pd.merge(stai_clean, id_mapping, on='id', how='inner')

panas_clean = panas.rename(columns={'user_id': 'id'})
panas_clean = pd.merge(panas_clean, id_mapping, on='id', how='inner')

print(f"✅ השאלונים נטענו בהצלחה! סה\"כ משתתפים שונים במחקר: {len(id_mapping)}")

מחלץ שאלונים מקובץ ה-ZIP...
✅ השאלונים נטענו בהצלחה! סה"כ משתתפים שונים במחקר: 53


In [3]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. מציאת ה-Baseline (היום הראשון) מתוך השאלונים בלבד!
# ---------------------------------------------------------
all_dates = pd.concat([
    stai_clean[['new_id', 'date']], 
    panas_clean[['new_id', 'date']]
])
baseline_dates = all_dates.groupby('new_id')['date'].min().reset_index(name='first_study_day')

# ---------------------------------------------------------
# 2. יצירת טבלת אגרגציה שבועית (Weekly Aggregation)
# ---------------------------------------------------------
stai_with_baseline = pd.merge(stai_clean, baseline_dates, on='new_id', how='left')
panas_with_baseline = pd.merge(panas_clean, baseline_dates, on='new_id', how='left')

stai_with_baseline['study_week'] = ((stai_with_baseline['date'] - stai_with_baseline['first_study_day']).dt.days // 7) + 1
panas_with_baseline['study_week'] = ((panas_with_baseline['date'] - panas_with_baseline['first_study_day']).dt.days // 7) + 1

weekly_stai = stai_with_baseline.groupby(['new_id', 'study_week'])['stai_stress'].mean().reset_index()
weekly_panas = panas_with_baseline.groupby(['new_id', 'study_week'])[['positive_affect_score', 'negative_affect_score']].mean().reset_index()

weekly_surveys_merged = pd.merge(weekly_stai, weekly_panas, on=['new_id', 'study_week'], how='outer')
weekly_surveys_merged = pd.merge(weekly_surveys_merged, baseline_dates, on='new_id', how='left')

weekly_surveys_df = pd.merge(weekly_surveys_merged, id_mapping, on='new_id', how='left')
ordered_columns = ['id', 'new_id', 'study_week', 'stai_stress', 'positive_affect_score', 'negative_affect_score']
weekly_surveys_df = weekly_surveys_df[ordered_columns].sort_values(by=['new_id', 'study_week']).reset_index(drop=True)

# ---------------------------------------------------------
# 3. דו"ח סטטיסטי קליני (EDA)
# ---------------------------------------------------------
print("==================================================")
print("     דו\"ח סטטיסטי קליני: נתונים גולמיים (EDA)")
print("==================================================\n")

print("--- 1. נתונים כלליים: שאלונים זמינים ---")
total_users = len(id_mapping)
stai_count = len(stai_clean)
panas_count = len(panas_clean)

print(f"סך הכל נבדקים שונים עם שאלונים: {total_users}")
print(f"סך הכל שאלוני STAI שמולאו: {stai_count}")
print(f"סך הכל שאלוני PANAS שמולאו: {panas_count}")
print(f"ממוצע שאלוני STAI לנבדק: {stai_count / total_users:.2f}")
print(f"ממוצע שאלוני PANAS לנבדק: {panas_count / total_users:.2f}\n")

print("--- 2. ניתוח רציפות ומרווחים (Gaps Analysis) ---")
combined_surveys_df = pd.concat([
    stai_clean[['new_id', 'date']].assign(type='STAI'),
    panas_clean[['new_id', 'date']].assign(type='PANAS')
]).sort_values(by=['new_id', 'date'])

combined_surveys_df['days_since_last'] = combined_surveys_df.groupby('new_id')['date'].diff().dt.days
survey_gaps_df = combined_surveys_df[combined_surveys_df['days_since_last'] > 10]
users_with_gaps_count = survey_gaps_df['new_id'].nunique()

print(f"מספר הפעמים הכולל שנרשם פער של יותר מ-10 ימים: {len(survey_gaps_df)}")
print(f"מספר הנבדקים שחוו פער כזה: {users_with_gaps_count} מתוך {total_users}")
if not survey_gaps_df.empty:
    print(f"המרווח הגדול ביותר ללא מענה נרשם על: {combined_surveys_df['days_since_last'].max():.0f} ימים.\n")

print("--- 3. ניתוח רגשות קיצוניים (PANAS ו-STAI) ---")
panas_threshold = 35
stai_threshold = 50

extreme_panas_df = panas_clean[panas_clean['negative_affect_score'] >= panas_threshold]
extreme_stai_df = stai_clean[stai_clean['stai_stress'] >= stai_threshold]

print(f"[PANAS] סה\"כ דיווחים על רגש שלילי גבוה ({panas_threshold}+): {len(extreme_panas_df)}")
if len(extreme_panas_df) > 0:
    print(f"        דווח על ידי {extreme_panas_df['new_id'].nunique()} נבדקים שונים.")

print(f"[STAI] סה\"כ דיווחים על חרדה ולחץ קיצוני ({stai_threshold}+): {len(extreme_stai_df)}")
if len(extreme_stai_df) > 0:
    print(f"        דווח על ידי {extreme_stai_df['new_id'].nunique()} נבדקים שונים.\n")

print("--- 4. שורות זמינות לחישוב Delta ---")
past_weeks_df = weekly_surveys_df.copy()
past_weeks_df['match_week'] = past_weeks_df['study_week'] + 1
past_weeks_df = past_weeks_df.rename(columns={'stai_stress': 'prev_stai_stress'})

weekly_pairs_df = pd.merge(
    weekly_surveys_df,
    past_weeks_df[['new_id', 'match_week', 'prev_stai_stress']],
    left_on=['new_id', 'study_week'],
    right_on=['new_id', 'match_week'],
    how='inner'
)
print(f"סה\"כ תצפיות רציפות (שבוע מול שבוע קודם) שזמינות לאימון: {len(weekly_pairs_df)}")

     דו"ח סטטיסטי קליני: נתונים גולמיים (EDA)

--- 1. נתונים כלליים: שאלונים זמינים ---
סך הכל נבדקים שונים עם שאלונים: 53
סך הכל שאלוני STAI שמולאו: 279
סך הכל שאלוני PANAS שמולאו: 268
ממוצע שאלוני STAI לנבדק: 5.26
ממוצע שאלוני PANAS לנבדק: 5.06

--- 2. ניתוח רציפות ומרווחים (Gaps Analysis) ---
מספר הפעמים הכולל שנרשם פער של יותר מ-10 ימים: 69
מספר הנבדקים שחוו פער כזה: 39 מתוך 53
המרווח הגדול ביותר ללא מענה נרשם על: 111 ימים.

--- 3. ניתוח רגשות קיצוניים (PANAS ו-STAI) ---
[PANAS] סה"כ דיווחים על רגש שלילי גבוה (35+): 18
        דווח על ידי 11 נבדקים שונים.
[STAI] סה"כ דיווחים על חרדה ולחץ קיצוני (50+): 124
        דווח על ידי 37 נבדקים שונים.

--- 4. שורות זמינות לחישוב Delta ---
סה"כ תצפיות רציפות (שבוע מול שבוע קודם) שזמינות לאימון: 145
